# 🎬 Rick's Clips — Free Google Colab GPU Renderer (Nvidia T4 16GB)
Renders all 1,821 frames of your episode at full 1080p (1080x1920) in minutes using Google's free cloud GPU.

---
### ⚡ Quick Start (2 Steps):
1. **Enable Free GPU**: Top menu -> `Runtime` -> `Change runtime type` -> select **T4 GPU** -> click `Save`.
2. **Click Run**: Click the **▶ Play button** on the cell below.

When rendering finishes, your MP4 video will automatically download to your computer.

In [ ]:
#@title 🚀 1-Click Render: Select Episode & Click Play
import os, sys, time
from google.colab import files

EPISODE = "boston-molasses" #@param ["boston-molasses", "lake-peigneur"]
SCALE = 1.0 #@param {type:"slider", min:0.5, max:1.0, step:0.1}

print("="*65)
print(f"🎬 STARTING CLOUD GPU RENDER: {EPISODE.upper()} (Scale: {SCALE})")
print("="*65)

# 1. Verify GPU
print("\n[1/5] 🔍 Checking GPU...")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

# 2. Install Dependencies
print("\n[2/5] 📦 Installing Node.js 20, FFmpeg, Xvfb & system libraries...")
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
!apt-get install -y nodejs ffmpeg xvfb libasound2 libgbm1 libnss3 > /dev/null 2>&1

# 3. Clone Repository & Install Playwright
print("\n[3/5] 📥 Pulling episode project from GitHub...")
if not os.path.exists("/content/ricks-clips"):
    !git clone https://github.com/pradeep142536/ricks-clips-00.git /content/ricks-clips
else:
    !git -C /content/ricks-clips pull origin main

print("       Installing Playwright Chromium engine...")
!cd /content/ricks-clips/templates/threejs_film && npm install playwright > /dev/null 2>&1
!npx playwright install --with-deps chromium > /dev/null 2>&1

# 4. Bake Film Plan
print(f"\n[4/5] 📐 Baking film plan for {EPISODE}...")
%cd /content/ricks-clips
!python tools/film_plan.py check episodes/{EPISODE}/film.json

# 5. Render Full Video with Hardware Acceleration
print(f"\n[5/5] 🎥 Rendering full 1080p video frames...")
t0 = time.time()
out_file = f"/content/{EPISODE}_full_episode_1080p.mp4"
!xvfb-run -s "-screen 0 1080x1920x24" node templates/threejs_film/render.mjs episodes/{EPISODE}/film.json --animatic --scale {SCALE} --out {out_file}
elapsed = time.time() - t0

# 6. Verify and Download to PC
if os.path.exists(out_file) and os.path.getsize(out_file) > 1000:
    size_mb = os.path.getsize(out_file) / (1024 * 1024)
    print("="*65)
    print(f"🎉 SUCCESS! Full 1080p video rendered in {elapsed:.1f}s ({elapsed/60:.1f} min)!")
    print(f"📁 Output File: {out_file} ({size_mb:.2f} MB)")
    print("⬇️ Initiating automatic browser download to your PC...")
    print("="*65)
    try:
        files.download(out_file)
    except Exception as e:
        print("Download trigger notice (can also download from left file panel):", e)
else:
    print("\n❌ Error: Video generation failed. Check logs above.")
